# Imports

In [29]:
import pandas as pd
from lyricsgenius import Genius
from pathlib import Path
import networkx as nx

# Data retrieval and cleaning

In [21]:
# Load raw data
tracks_df = pd.read_csv("dataset.csv")

# Drop old index column if it exists
if "Unnamed: 0" in tracks_df.columns:
    tracks_df = tracks_df.drop(columns=["Unnamed: 0"])

# Remove songs that share the same title (keep the first occurrence)
tracks_df = tracks_df.drop_duplicates(subset="track_name", keep="first")

# Discard any rows that contain missing values in *any* column
# tracks_df = tracks_df.dropna(axis=0, how="any")

# Binarize the explicit flag from boolean to integer (0 or 1)
tracks_df["explicit"] = tracks_df["explicit"].map({True: 1, False: 0})


# Create a helper field for searching: "Track name - artist1, artist2, ..."
def make_search_string(row):
    artist_list = [a.strip() for a in str(row["artists"]).split(";")]
    return f'{row["track_name"]} - {", ".join(artist_list)}'


tracks_df["search_string"] = tracks_df.apply(make_search_string, axis=1)

# Save cleaned version to disk
tracks_df.to_csv("dataset_clean.csv", index=False)

# Inspect the first few feature columns (adjust indices if needed)
# tracks_df.columns.values[4:19]


# Summary statistics for cleaned dataset

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


def top_artists(df, n=10):
    # artists are semicolon-separated; return top individual artists
    if "artists" not in df.columns:
        return pd.Series(dtype=int)
    return (
        df["artists"]
        .astype(str)
        .str.split(";")
        .explode()
        .str.strip()
        .value_counts()
        .head(n)
    )


def summarize(df, name):
    print(f"===== Summary for {name} =====")
    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print("\nDtype counts:")
    print(df.dtypes.value_counts())
    print("\nMissing values (top 20):")
    print(df.isna().sum().sort_values(ascending=False).head(20))
    print(
        "\nDuplicate track_name count:",
        (
            df.duplicated(subset="track_name").sum()
            if "track_name" in df.columns
            else "N/A"
        ),
    )
    print(
        "Unique track_name:",
        df["track_name"].nunique() if "track_name" in df.columns else "N/A",
    )
    print("\nTop 10 track_genre (if present):")
    if "track_genre" in df.columns:
        print(df["track_genre"].value_counts().head(10))
    else:
        print("N/A")
    print("\nTop 10 artists:")
    print(top_artists(df, 10))
    print("\nNumeric describe (selected):")
    print(
        df.select_dtypes(include="number")
        .describe()
        .transpose()
        .loc[:, ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    )
    print("\nSample rows (5):")
    # display(df.head(5))
    print("\n")


# Run summaries
summarize(tracks_df, "dataset_clean.csv (clean)")

===== Summary for dataset_clean.csv (clean) =====
Shape: (73609, 21)
Columns: 21

Dtype counts:
float64    9
object     6
int64      6
Name: count, dtype: int64

Missing values (top 20):
album_name          1
track_name          1
artists             1
track_id            0
speechiness         0
track_genre         0
time_signature      0
tempo               0
valence             0
liveness            0
instrumentalness    0
acousticness        0
loudness            0
mode                0
key                 0
energy              0
danceability        0
explicit            0
duration_ms         0
popularity          0
dtype: int64

Duplicate track_name count: 0
Unique track_name: 73608

Top 10 track_genre (if present):
track_genre
black-metal    981
comedy         965
afrobeat       945
heavy-metal    944
cantopop       935
bluegrass      930
forro          925
anime          925
grindcore      924
malay          921
Name: count, dtype: int64

Top 10 artists:
artists
Pritam           

## Average feature scores by genre
Compute mean values for all numeric audio features grouped by `track_genre`.

In [ ]:
# Aggregate numeric feature columns by genre but exclude numeric columns at positions 1, 2 and the last
numeric_feature_cols = tracks_df.select_dtypes(include="number").columns.tolist()

# Exclude by position (0-based): drop indices 0,1 and the last one
#cols_to_use = numeric_feature_cols[2:-1]

genre_feature_means = (
    tracks_df.dropna(subset=["track_genre"])
    .groupby("track_genre")[numeric_feature_cols]
    .mean()
    .sort_index()
)

# Preview
genre_feature_means.round(3)

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
track_genre,,,,,,,,,,,,,,,
acoustic,43.246,214887.266,0.056,0.551,0.440,5.098,-9.443,0.813,0.043,0.560,0.039,0.155,0.427,118.977,3.894
afrobeat,24.590,248380.608,0.017,0.668,0.702,5.547,-7.808,0.506,0.086,0.267,0.260,0.183,0.698,118.712,3.948
alt-rock,38.803,236604.360,0.057,0.538,0.743,5.641,-6.349,0.625,0.055,0.134,0.056,0.216,0.517,124.332,3.943
alternative,35.871,211357.085,0.317,0.612,0.691,5.442,-6.101,0.540,0.094,0.183,0.019,0.207,0.488,119.572,3.929
ambient,44.929,236095.845,0.006,0.366,0.235,4.969,-18.765,0.615,0.042,0.776,0.683,0.128,0.166,110.568,3.648
anime,48.736,209012.732,0.058,0.538,0.668,5.319,-8.120,0.504,0.089,0.275,0.274,0.197,0.431,123.269,3.922
black-metal,22.409,310430.977,0.131,0.295,0.875,5.398,-6.528,0.590,0.087,0.027,0.444,0.241,0.191,128.786,3.828
bluegrass,25.743,222260.235,0.005,0.535,0.532,5.513,-9.913,0.853,0.039,0.558,0.136,0.229,0.638,126.580,3.909
blues,39.160,236325.922,0.018,0.566,0.597,5.292,-8.359,0.680,0.063,0.324,0.044,0.179,0.575,119.401,3.883


In [ ]:
def load_env_file(path="api.env"):
    """
    Very small .env-style loader.
    Returns a dict like {"LASTFM_API_KEY": "...", "LASTFM_API_SECRET": "..."}.
    """
    env_vars = {}
    env_path = Path(path)

    if not env_path.exists():
        raise FileNotFoundError(f"{path} not found in current folder")

    with env_path.open() as f:
        for line in f:
            line = line.strip()
            # skip empty lines and comments
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_vars[key.strip()] = value.strip()

    return env_vars


env = load_env_file(".env.local")
GENIUS_API_KEY = env.get("GENIUS_ACCESS_TOKEN")

if not GENIUS_API_KEY:
    raise ValueError("GENIUS_API_KEY not found or empty in .env.local")

In [25]:
# Retrieve API access token
genius = Genius(GENIUS_API_KEY)

# Search for top three songs of artist
artist = genius.search_artist("Kendrick Lamar", max_songs=3)
song = genius.search_song("Not Like Us", artist.name)

artist.save_lyrics()

# print(song.lyrics)


Searching for songs by Kendrick Lamar...

Song 1: "Not Like Us"
Song 2: "HUMBLE."
Song 3: "euphoria"

Reached user-specified song limit (3).
Done. Found 3 songs.
Searching for "Not Like Us" by Kendrick Lamar...
Done.
Wrote saved_artist_lyrics_kendrick_lamar_3_songs.json.
